In [0]:
# Step 1 - Import Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Step 2 - Use Catalog & Schema
spark.sql("USE CATALOG Bronze_Catalog")
spark.sql("USE SCHEMA Bronze_SCH")

In [0]:
# Step 3 - Read Parquet File
device_df = spark.read.parquet(
    "abfss://pro2@adlsstgacntp08.dfs.core.windows.net/parquet/device_metrics_stream_v2.parquet"
)

In [0]:
# Step 4 - Check Schema
device_df.printSchema()

In [0]:
# Step 5 - Check Sample Data
display(device_df.limit(10))

In [0]:
print("Total Records :", device_df.count())

In [0]:
from pyspark.sql.functions import current_timestamp

device_df = device_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [0]:
spark.sql("""
SHOW TABLES IN Bronze_Catalog.Bronze_SCH
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
""").show(truncate=False)

In [0]:
last_watermark = spark.sql("""
SELECT last_processed_timestamp
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name = 'Bronze_Device_Metrics'
""").collect()[0][0]

print(last_watermark)

In [0]:
incremental_df = device_df.filter(
    col("ingestion_timestamp") > last_watermark
)

records_loaded = incremental_df.count()

print(f"New Records : {records_loaded}")

display(incremental_df.limit(10))

In [0]:
# Create a dummy ingestion timestamp during Bronze loading:
from pyspark.sql.functions import current_timestamp

device_df = device_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [0]:
device_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp

try:

    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema","true") \
        .saveAsTable("Bronze_Catalog.Bronze_SCH.Bronze_Device_Metrics")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Device_Metrics',
            'Device Bronze Pipeline',
            current_timestamp(),
            {records_loaded},
            'SUCCESS',
            NULL
        )
    """)

    print(f"✅ Bronze_Device_Metrics loaded successfully.Records Loaded : {records_loaded}")
    

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Device_Metrics',
            'Device Bronze Pipeline',
            current_timestamp(),
            0,
            'FAILED',
            '{error_message}'
        )
    """)

    print("❌ Bronze_Device_Metrics Pipeline Failed")
    print(f"⚠️ Error : {error_message}")
    

    raise

In [0]:
spark.sql("""
UPDATE Bronze_Catalog.Bronze_SCH.Watermark_Table
SET last_processed_timestamp = (
    SELECT MAX(ingestion_timestamp)
    FROM Bronze_Catalog.Bronze_SCH.Bronze_Device_Metrics
)
WHERE table_name='Bronze_Device_Metrics'
""")

print("✅ Watermark Updated Successfully")

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name='Bronze_Device_Metrics'
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Audit_Log
WHERE table_name='Bronze_Device_Metrics'
ORDER BY load_time DESC
LIMIT 5
""").show(truncate=False)